# Multi-Stage Movie Recommender: Tutorial

This notebook is a compact, runnable learning reference for the multi-stage recommendation pipeline.

Recommended workflow for students:
1. Run all cells once from top to bottom.
2. Re-read each section and connect the equations to the code.
3. Re-run with one hyperparameter changed (for example, factor size or candidate pool size) and observe metric/runtime changes.

## Learning Objectives

1. Build a two-stage recommender pipeline: candidate generation then reranking.
2. Understand why chronological splitting is required for realistic offline evaluation.
3. Interpret ranking metrics and latency-quality tradeoffs.
4. Practice mapping concepts into a different code interface for assignment work.

By the end, you should be able to explain both what the model does and why each stage is needed.

## How to use this notebook

This tutorial intentionally uses a compact API so the architecture is easy to inspect. Differences from the end-to-end experiment include:
- Function names and variable names differ.
- Reporting table schema differs.
- Some helper functions here combine multiple steps that are separated in the assignment.

Run the cells from top to bottom, then change one hyperparameter at a time and observe the quality/runtime trade-off.

In [ ]:
import os
import time
import random
import numpy as np
import pandas as pd
from scipy import sparse
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)


In [ ]:
# Tutorial config (kept compact for classroom runtime).
cfg = {
    'implicit_threshold': 4.0,
    'min_user_events': 10,
    'max_users_for_demo': 3000,
    'val_ratio': 0.2,
    'test_ratio': 0.2,
    'als_factors': 24,
    'als_iters': 3,
    'als_alpha': 20.0,
    'als_reg': 0.1,
    'candidate_pool_size': 150,
    'final_k': 10,
    'tower_emb_dim': 24,
    'tower_hidden': 48,
    'tower_epochs': 2,
    'tower_batch': 1024,
    'max_positive_pairs': 50000,
    'neg_per_pos': 1
}
cfg


## 1) Load Data and Build Interaction Table
Tutorial API here differs from assignment API. In assignment, data preparation steps are spread across separate cells/functions.


In [ ]:
GENRE_COLS_100K = [
    'unknown', 'Action', 'Adventure', 'Animation', "Children's", 'Comedy', 'Crime',
    'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery',
    'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western'
]

def load_interactions_and_catalog():
    r20 = os.path.join('data', 'ml-20m', 'ratings.csv')
    m20 = os.path.join('data', 'ml-20m', 'movies.csv')
    if os.path.exists(r20) and os.path.exists(m20):
        interactions = pd.read_csv(r20)
        catalog = pd.read_csv(m20)[["movieId", "title", "genres"]]
        return interactions, catalog, 'ml-20m'

    r100 = os.path.join('..', 'assignment 1', 'ml-100k', 'u.data')
    i100 = os.path.join('..', 'assignment 1', 'ml-100k', 'u.item')
    if os.path.exists(r100) and os.path.exists(i100):
        interactions = pd.read_csv(r100, sep='\t', header=None, names=['userId', 'movieId', 'rating', 'timestamp'])
        names = ['movieId', 'title', 'release_date', 'video_release_date', 'IMDb_URL'] + GENRE_COLS_100K
        raw = pd.read_csv(i100, sep='|', header=None, names=names, encoding='latin-1')
        def row_genres(row):
            active = [g for g in GENRE_COLS_100K if row[g] == 1]
            return '|'.join(active) if active else '(no genres listed)'
        catalog = raw[["movieId", "title"]].copy()
        catalog['genres'] = raw.apply(row_genres, axis=1)
        return interactions, catalog, 'ml-100k-fallback'

    raise FileNotFoundError('Could not find ml-20m or fallback ml-100k data.')

interactions_raw, item_catalog, source_name = load_interactions_and_catalog()
print('Data source:', source_name)
print('Raw rows:', len(interactions_raw))
print('Unique users/items:', interactions_raw.userId.nunique(), interactions_raw.movieId.nunique())


In [ ]:
def build_demo_interactions(df, threshold, min_user_events, max_users, seed=42):
    work = df.copy()
    if 'timestamp' not in work.columns:
        work['timestamp'] = np.arange(len(work), dtype=np.int64)

    # Keep only users with enough history to support temporal split.
    user_counts = work.groupby("userId").size()
    eligible = user_counts[user_counts >= min_user_events].index.values
    rng = np.random.default_rng(seed)
    if len(eligible) > max_users:
        chosen = rng.choice(eligible, size=max_users, replace=False)
    else:
        chosen = eligible

    work = work[work["userId"].isin(chosen)].copy()
    work = work.sort_values(["userId", "timestamp"])
    work["label"] = (work["rating"] >= threshold).astype(np.float32)
    work = work[work["label"] > 0].copy()

    # Remap IDs to dense [0..N-1] indices for matrix operations.
    uid_vals = np.sort(work.userId.unique())
    iid_vals = np.sort(work.movieId.unique())
    uid_map = {u: i for i, u in enumerate(uid_vals)}
    iid_map = {m: i for i, m in enumerate(iid_vals)}

    work["uidx"] = work["userId"].map(uid_map).astype(int)
    work["iidx"] = work["movieId"].map(iid_map).astype(int)

    return work, uid_map, iid_map

interactions, user_index, item_index = build_demo_interactions(
    interactions_raw,
    threshold=cfg['implicit_threshold'],
    min_user_events=cfg['min_user_events'],
    max_users=cfg['max_users_for_demo'],
    seed=SEED
)
print('Demo users/items/interactions:', len(user_index), len(item_index), len(interactions))

## 3) Chronological Holdout and Sparse Feedback Matrix

Why chronological split?
- Recommendation is a temporal prediction problem.
- Training must only use past interactions, and validation/test should mimic future interactions.

In this tutorial, each user history is split in timestamp order into train, validation, and test.

Matrix construction:
- We build a sparse user-item matrix $R \in \mathbb{R}^{m \times n}$ where non-zero entries indicate implicit positive feedback.
- Sparse storage is critical because most user-item pairs are missing (extreme sparsity).

$$
R_{ui} = \begin{cases}
1 & \text{if user } u \text{ interacted positively with item } i\\
0 & \text{otherwise}
\end{cases}
$$

Migration hint: the assignment requests similar logic but with different function boundaries and naming.

In [ ]:
def chronological_holdout(df, val_ratio=0.2, test_ratio=0.2):
    tr, va, te = [], [], []
    for _, g in df.sort_values("timestamp").groupby("uidx"):
        n = len(g)
        if n < 5:
            tr.append(g)
            continue
        n_test = max(1, int(round(n * test_ratio)))
        n_val = max(1, int(round(n * val_ratio)))
        s1 = n - (n_val + n_test)
        s2 = n - n_test
        if s1 <= 0:
            s1 = max(1, n - 2)
            s2 = n - 1
        tr.append(g.iloc[:s1])
        va.append(g.iloc[s1:s2])
        te.append(g.iloc[s2:])
    return (
        pd.concat(tr, ignore_index=True),
        pd.concat(va, ignore_index=True) if len(va) else pd.DataFrame(columns=df.columns),
        pd.concat(te, ignore_index=True) if len(te) else pd.DataFrame(columns=df.columns),
    )

def as_feedback_matrix(df, n_users, n_items):
    if len(df) == 0:
        return sparse.csr_matrix((n_users, n_items), dtype=np.float32)
    return sparse.csr_matrix((df.label.values.astype(np.float32), (df.uidx.values, df.iidx.values)), shape=(n_users, n_items))

train_tbl, val_tbl, test_tbl = chronological_holdout(
    interactions,
    val_ratio=cfg['val_ratio'],
    test_ratio=cfg['test_ratio']
)

n_users = len(user_index)
n_items = len(item_index)
train_mat = as_feedback_matrix(train_tbl, n_users, n_items)
val_mat = as_feedback_matrix(val_tbl, n_users, n_items)
test_mat = as_feedback_matrix(test_tbl, n_users, n_items)
print(train_mat.shape, train_mat.nnz, val_mat.nnz, test_mat.nnz)


In [ ]:
def rec_at_k(pred, truth, k):
    if len(truth) == 0:
        return np.nan
    return len(set(pred[:k]).intersection(set(truth))) / max(1, len(truth))

def hit_at_k(pred, truth, k):
    if len(truth) == 0:
        return np.nan
    return 1.0 if len(set(pred[:k]).intersection(set(truth))) > 0 else 0.0

def ndcg_at_k(pred, truth, k):
    if len(truth) == 0:
        return np.nan
    dcg = 0.0
    for rank, item in enumerate(pred[:k], start=1):
        if item in truth:
            dcg += 1.0 / np.log2(rank + 1)
    ideal_hits = min(k, len(truth))
    idcg = np.sum([1.0 / np.log2(r + 1) for r in range(1, ideal_hits + 1)])
    return dcg / idcg if idcg > 0 else np.nan

def score_ranking_dict(rankings, ground_truth_csr, k):
    recs, hits, nds = [], [], []
    for u, pred in rankings.items():
        truth = ground_truth_csr[u].indices.tolist()
        r = rec_at_k(pred, truth, k)
        h = hit_at_k(pred, truth, k)
        n = ndcg_at_k(pred, truth, k)
        if not np.isnan(r):
            recs.append(r)
            hits.append(h)
            nds.append(n)
    return {
        f'recall_{k}': float(np.mean(recs)) if recs else np.nan,
        f'hitrate_{k}': float(np.mean(hits)) if hits else np.nan,
        f'ndcg_{k}': float(np.mean(nds)) if nds else np.nan,
        'n_users_eval': len(recs)
    }


### Metric Notes (Read Before Running Models)

This tutorial reports Recall, HitRate, and NDCG.

Recall at K
$$
\mathrm{Recall@K}(u) = \frac{|\mathrm{TopK}(u) \cap \mathcal{T}_u|}{|\mathcal{T}_u|}
$$

HitRate at K
$$
\mathrm{HitRate@K}(u) = \mathbb{1}\{\mathrm{TopK}(u) \cap \mathcal{T}_u \neq \emptyset\}
$$

NDCG at K rewards placing relevant items earlier in ranked results.
Use all three metrics together to understand both coverage and ordering quality.

## 4) Stage-1 Candidate Generation with Implicit MF (ALS)

Goal of Stage-1: retrieve a manageable candidate set quickly.

Matrix factorization intuition:
- Learn dense latent vectors for users and items.
- Score by dot product in latent space.

$$
\hat{r}_{ui} = p_u^\top q_i
$$

For implicit ALS, confidence on observed positives is weighted higher than unobserved entries.
A common confidence form is:
$$
c_{ui} = 1 + \alpha r_{ui}
$$

Stage-1 output is top-$N$ candidates per user (here, candidate_pool_size).
This stage prioritizes recall and speed, not perfect ranking order.

In [ ]:
def recommend_by_frequency(train_csr, topk=10):
    freq = np.asarray(train_csr.sum(axis=0)).ravel()
    order = np.argsort(-freq)
    out = {}
    for u in range(train_csr.shape[0]):
        seen = set(train_csr[u].indices.tolist())
        out[u] = [i for i in order if i not in seen][:topk]
    return out

pop_top10 = recommend_by_frequency(train_mat, topk=cfg["final_k"])
pop_scores = score_ranking_dict(pop_top10, test_mat, cfg["final_k"])
pop_scores


In [ ]:
def fit_implicit_factors(csr, n_factors=24, alpha=20.0, reg=0.1, n_iters=3, seed=42):
    m, n = csr.shape
    rng = np.random.default_rng(seed)
    user_f = rng.normal(0, 0.1, size=(m, n_factors)).astype(np.float32)
    item_f = rng.normal(0, 0.1, size=(n, n_factors)).astype(np.float32)
    csr_t = csr.T.tocsr()
    I = np.eye(n_factors, dtype=np.float32)

    for _ in range(n_iters):
        # User update with fixed items.
        yty = item_f.T @ item_f + reg * I
        for u in range(m):
            idx = csr[u].indices
            if len(idx) == 0:
                continue
            Y = item_f[idx]
            c = 1.0 + alpha * np.ones(len(idx), dtype=np.float32)
            A = yty + (Y.T * (c - 1.0)) @ Y
            b = (Y.T * c) @ np.ones(len(idx), dtype=np.float32)
            user_f[u] = np.linalg.solve(A, b)

        # Item update with fixed users (symmetric step).
        xtx = user_f.T @ user_f + reg * I
        for i in range(n):
            idx = csr_t[i].indices
            if len(idx) == 0:
                continue
            X = user_f[idx]
            c = 1.0 + alpha * np.ones(len(idx), dtype=np.float32)
            A = xtx + (X.T * (c - 1.0)) @ X
            b = (X.T * c) @ np.ones(len(idx), dtype=np.float32)
            item_f[i] = np.linalg.solve(A, b)

    return user_f, item_f

def make_candidate_pool(user_f, item_f, train_csr, pool_size=150):
    score_mat = user_f @ item_f.T
    pools = {}
    for u in range(score_mat.shape[0]):
        row = score_mat[u].copy()
        # Never recommend items already seen in training history.
        row[train_csr[u].indices] = -1e9
        if pool_size >= len(row):
            pools[u] = np.argsort(-row).tolist()
            continue
        idx = np.argpartition(-row, pool_size)[:pool_size]
        idx = idx[np.argsort(-row[idx])]
        pools[u] = idx.tolist()
    return pools

t0 = time.time()
user_factors, item_factors = fit_implicit_factors(
    train_mat,
    n_factors=cfg['als_factors'],
    alpha=cfg['als_alpha'],
    reg=cfg['als_reg'],
    n_iters=cfg['als_iters'],
    seed=SEED
)
als_time = time.time() - t0
candidates = make_candidate_pool(user_factors, item_factors, train_mat, pool_size=cfg["candidate_pool_size"])
mf_top10 = {u: items[:cfg["final_k"]] for u, items in candidates.items()}
mf_scores_10 = score_ranking_dict(mf_top10, test_mat, cfg["final_k"])
mf_scores_pool = score_ranking_dict(candidates, test_mat, cfg["candidate_pool_size"])
print('ALS time (sec):', round(als_time, 2))
mf_scores_10, mf_scores_pool

## 5) Stage-2 Two-Tower Reranking with Side Features

Goal of Stage-2: improve ordering quality inside the Stage-1 candidate pool.

Two-tower scoring:
- User tower maps user features to an embedding $f_u(x_u)$.
- Item tower maps item features to an embedding $g_i(x_i)$.
- Relevance score is dot product (or cosine-style after normalization).

$$
s(u,i) = f_u(x_u)^\top g_i(x_i)
$$

We train with positives and sampled negatives using binary cross-entropy on logits.
This stage is usually slower than retrieval but still efficient over a small candidate set.

Migration hint: assignment asks for similar concepts with different function and variable names.

In [ ]:
def build_item_genre_features(catalog_df, iid_map, max_dims=64):
    part = catalog_df[catalog_df.movieId.isin(iid_map.keys())].copy()
    part['genres'] = part['genres'].fillna('(no genres listed)')
    vocab = sorted({g for s in part.genres.tolist() for g in str(s).split("|")})
    if len(vocab) > max_dims:
        vocab = vocab[:max_dims]
    gid = {g: i for i, g in enumerate(vocab)}
    Xg = np.zeros((len(iid_map), len(gid)), dtype=np.float32)
    for row in part.itertuples(index=False):
        i = iid_map[row.movieId]
        for g in str(row.genres).split("|"):
            if g in gid:
                Xg[i, gid[g]] = 1.0
    return Xg

item_side = build_item_genre_features(item_catalog, item_index, max_dims=64)
user_side = np.zeros((n_users, item_side.shape[1]), dtype=np.float32)
for u in range(n_users):
    seen = train_mat[u].indices
    if len(seen) > 0:
        user_side[u] = item_side[seen].mean(axis=0)

# Different from assignment naming intentionally.
user_input = np.hstack([user_factors, user_side]).astype(np.float32)
item_input = np.hstack([item_factors, item_side]).astype(np.float32)
user_input.shape, item_input.shape


In [ ]:
class DualEncoderRanker(nn.Module):
    def __init__(self, user_dim, item_dim, hidden=48, emb_dim=24):
        super().__init__()
        self.user_net = nn.Sequential(
            nn.Linear(user_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, emb_dim)
        )
        self.item_net = nn.Sequential(
            nn.Linear(item_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, emb_dim)
        )

    def forward(self, ux, ix):
        u = F.normalize(self.user_net(ux), dim=1)
        i = F.normalize(self.item_net(ix), dim=1)
        return (u * i).sum(dim=1)

def make_train_pairs(feedback_csr, max_pos=50000, neg_per_pos=1, seed=42):
    rng = np.random.default_rng(seed)
    pu, pi = feedback_csr.nonzero()
    if len(pu) > max_pos:
        pick = rng.choice(len(pu), size=max_pos, replace=False)
        pu, pi = pu[pick], pi[pick]
    seen_sets = [set(feedback_csr[u].indices.tolist()) for u in range(feedback_csr.shape[0])]
    uu, ii, yy = [], [], []
    for u, i in zip(pu, pi):
        uu.append(u); ii.append(i); yy.append(1.0)
        for _ in range(neg_per_pos):
            j = int(rng.integers(0, feedback_csr.shape[1]))
            while j in seen_sets[u]:
                j = int(rng.integers(0, feedback_csr.shape[1]))
            uu.append(u); ii.append(j); yy.append(0.0)
    return np.asarray(uu), np.asarray(ii), np.asarray(yy, dtype=np.float32)

def fit_reranker(user_x, item_x, train_csr, cfg, seed=42):
    uu, ii, yy = make_train_pairs(
        train_csr,
        max_pos=cfg['max_positive_pairs'],
        neg_per_pos=cfg['neg_per_pos'],
        seed=seed
    )
    model = DualEncoderRanker(
        user_dim=user_x.shape[1],
        item_dim=item_x.shape[1],
        hidden=cfg['tower_hidden'],
        emb_dim=cfg['tower_emb_dim']
    )
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.BCEWithLogitsLoss()

    ux = torch.from_numpy(user_x[uu])
    ix = torch.from_numpy(item_x[ii])
    yt = torch.from_numpy(yy)
    n = len(yy)
    batch = cfg['tower_batch']

    t0 = time.time()
    for ep in range(cfg['tower_epochs']):
        perm = torch.randperm(n)
        total = 0.0
        for s in range(0, n, batch):
            idx = perm[s:s+batch]
            logits = model(ux[idx], ix[idx])
            loss = loss_fn(logits, yt[idx])
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += float(loss.item()) * len(idx)
        print(f"epoch {ep + 1}/{cfg['tower_epochs']} loss={total / n:.4f}")
    train_time = time.time() - t0
    return model, train_time

@torch.no_grad()
def rerank_from_pool(pool_dict, model, user_x, item_x, topk=10):
    model.eval()
    out = {}
    for u, pool in pool_dict.items():
        if len(pool) == 0:
            out[u] = []
            continue
        p = np.asarray(pool, dtype=int)
        ux = torch.from_numpy(np.repeat(user_x[u:u+1], len(p), axis=0))
        ix = torch.from_numpy(item_x[p])
        s = model(ux, ix).cpu().numpy()
        ord_idx = np.argsort(-s)
        out[u] = [int(p[j]) for j in ord_idx[:topk]]
    return out

reranker, tower_time = fit_reranker(user_input, item_input, train_mat, cfg, seed=SEED)
print('Two-tower time (sec):', round(tower_time, 2))
reranked_top10 = rerank_from_pool(candidates, reranker, user_input, item_input, topk=cfg['final_k'])
rerank_scores = score_ranking_dict(reranked_top10, test_mat, cfg['final_k'])
rerank_scores


## 6) Comparison Table and Interpretation

This section summarizes system behavior.
When reading the table, focus on:
1. Quality: recall10, ndcg10, hit10
2. Candidate quality: recall_pool, hit_pool
3. Cost: train_sec

Typical pattern in multi-stage systems:
- Stage-1 improves retrieval coverage.
- Stage-2 may improve top-k ordering, but not always; effectiveness depends on feature quality and training signal.

In [ ]:
comparison = pd.DataFrame([
    {
        'model_name': 'popularity',
        'recall10': pop_scores['recall_10'],
        'ndcg10': pop_scores['ndcg_10'],
        'hit10': pop_scores['hitrate_10'],
        'recall_pool': np.nan,
        'hit_pool': np.nan,
        'train_sec': np.nan
    },
    {
        'model_name': 'stage1_mf',
        'recall10': mf_scores_10['recall_10'],
        'ndcg10': mf_scores_10['ndcg_10'],
        'hit10': mf_scores_10['hitrate_10'],
        'recall_pool': mf_scores_pool[f"recall_{cfg['candidate_pool_size']}"] ,
        'hit_pool': mf_scores_pool[f"hitrate_{cfg['candidate_pool_size']}"] ,
        'train_sec': als_time
    },
    {
        'model_name': 'stage1_plus_stage2',
        'recall10': rerank_scores['recall_10'],
        'ndcg10': rerank_scores['ndcg_10'],
        'hit10': rerank_scores['hitrate_10'],
        'recall_pool': mf_scores_pool[f"recall_{cfg['candidate_pool_size']}"] ,
        'hit_pool': mf_scores_pool[f"hitrate_{cfg['candidate_pool_size']}"] ,
        'train_sec': als_time + tower_time
    }
])
comparison


In [ ]:
# Mini experiment: candidate pool sensitivity (different from assignment ablation spec).
rows = []
for pool_n in [30, 80, 150]:
    cands_n = make_candidate_pool(user_factors, item_factors, train_mat, pool_size=pool_n)
    rer_n = rerank_from_pool(cands_n, reranker, user_input, item_input, topk=cfg["final_k"])
    s_n = score_ranking_dict(rer_n, test_mat, cfg["final_k"])
    rows.append({
        'pool_n': pool_n,
        'recall10': s_n['recall_10'],
        'ndcg10': s_n['ndcg_10'],
        'hit10': s_n['hitrate_10']
    })
pd.DataFrame(rows)


## 7) Self-Check Quiz (Before Running the Full Experiment)

Try answering these without looking back at the code first.

1. Why do we split interactions by time instead of random split in recommendation tasks?
2. What problem appears if you do not remove training-seen items before producing recommendations?
3. In Stage-1 retrieval, why might Recall@pool be high while NDCG@10 is only moderate?
4. What is the practical role of negative sampling in two-tower training?
5. If reranking does not improve Stage-1 top-10 metrics, name two likely causes.

Optional mini exercise:
- Re-run with a larger `candidate_pool_size` and explain how `recall10`, `ndcg10`, and training/inference cost changed.

<details>
<summary>Quick answer hints</summary>

- Q1: Random split can leak future behavior into training and overestimate offline performance.
- Q2: The model can recommend already-consumed items, inflating metrics and reducing real utility.
- Q3: Stage-1 retrieval focuses on coverage/recall, not precise top-position ordering.
- Q4: Negatives provide contrastive signal so the model learns to separate relevant from non-relevant items.
- Q5: Typical issues are weak side features, insufficient training signal, or a too-small/too-noisy candidate pool.

</details>

## 8) From Tutorial to Full Experiment

Use this checklist while porting concepts:
1. Data prep: map the tutorial split/matrix logic into assignment function structure.
2. Metrics: assignment expects explicit Recall@K, NDCG@K, and HitRate@K reporting fields.
3. Stage-1: map latent-factor training and candidate generation into assignment API.
4. Stage-2: map model definition, pair construction, training loop, and reranking into assignment cells.
5. Reporting: assignment requires its own table schema and question-specific analysis text.

Common pitfalls:
- Forgetting to remove already-seen items before recommendation.
- Mixing user/item index maps between train and test.
- Reporting only top-10 metrics without candidate-level metrics.

Suggested student exercise after this tutorial:
- Change one hyperparameter at a time (for example pool size, factor dimension, or epochs).
- Write 2-3 lines explaining why each change affected quality and runtime.